# Week 6 Capstone — End-to-End Secondary-Structure Prediction

Loads a **saved** CNN checkpoint + config and predicts per-residue Q3 (H/E/C) for every chain in `test.fasta`.
No training happens here. Runtime → **Run all** reproduces `predictions.csv` and `predictions_smoothed.csv`.

**Artefacts location:** upload the `artefacts/` folder (and `test.fasta`) next to this notebook, or mount Drive and set `ART` below.

## 1. Setup & seeds

In [ ]:
import os, json, csv, random, hashlib
import numpy as np
import torch, torch.nn as nn

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.use_deterministic_algorithms(True, warn_only=True)
print('seeds set | torch', torch.__version__, '| cuda', torch.cuda.is_available())

## 2. Load artefacts (no training)

In [ ]:
ART = 'artefacts'   # <- change if artefacts live elsewhere (e.g. '/content/drive/MyDrive/week6/artefacts')
for f in ('cnn.pt', 'config.json', 'test.fasta'):
    assert os.path.exists(os.path.join(ART, f)) or (f == 'test.fasta' and os.path.exists(f)), \
        f'missing {f} — upload the artefacts/ folder and test.fasta'
CFG = json.load(open(os.path.join(ART, 'config.json')))
print('loaded config:', {k: CFG[k] for k in ('arch','num_chains','epochs','best_val_q3','test_q3') if k in CFG})

## 3. Vocabulary, model, and post-processing (self-contained)

In [ ]:
AA = 'ACDEFGHIKLMNPQRSTVWY'
AA_IDX = {a: i for i, a in enumerate(AA)}
UNK_IDX, PAD_IDX, VOCAB = 20, 21, 22
Q3_LABELS = {0: 'H', 1: 'E', 2: 'C'}
N_CLASSES = 3
SS8_TO_Q3 = {'H':'H','G':'H','I':'H','E':'E','B':'E','T':'C','S':'C','C':'C','-':'C','P':'C',' ':'C'}

def encode_sequence(seq):
    return torch.tensor([AA_IDX.get(c, UNK_IDX) for c in seq], dtype=torch.long)

def collapse_q3(ss):
    return ''.join(SS8_TO_Q3.get(c, 'C') for c in ss)

class CNN(nn.Module):
    def __init__(self, d=64, hidden=128, layers=3, kernel=7, dropout=0.2):
        super().__init__()
        self.emb = nn.Embedding(VOCAB, d, padding_idx=PAD_IDX)
        blocks, in_ch = [], d
        for _ in range(layers):
            blocks += [nn.Conv1d(in_ch, hidden, kernel, padding=kernel//2), nn.ReLU(), nn.Dropout(dropout)]
            in_ch = hidden
        self.body = nn.Sequential(*blocks)
        self.head = nn.Conv1d(hidden, N_CLASSES, 1)
    def forward(self, x):
        h = self.emb(x).transpose(1, 2)
        h = self.body(h)
        return self.head(h).transpose(1, 2)

def smooth_predictions(pred, min_segment=3):
    labels = list(pred); changed = True
    while changed:
        changed = False; i = 0
        while i < len(labels):
            cls = labels[i]; j = i
            while j < len(labels) and labels[j] == cls: j += 1
            if (j - i) < min_segment:
                rep = labels[i-1] if i > 0 else (labels[j] if j < len(labels) else cls)
                if rep != cls:
                    for k in range(i, j): labels[k] = rep
                    changed = True
            i = j
    return ''.join(labels)

## 4. The `SecStructPredictor` pipeline class

In [ ]:
def read_fasta(path):
    entries, pid, buf = [], None, []
    for line in open(path):
        line = line.strip()
        if not line: continue
        if line.startswith('>'):
            if pid is not None: entries.append((pid, ''.join(buf)))
            pid, buf = line[1:].split()[0], []
        else: buf.append(line)
    if pid is not None: entries.append((pid, ''.join(buf)))
    return entries

class SecStructPredictor:
    def __init__(self, model, config, device='cpu'):
        self.model, self.config, self.device = model, config, device
        self.model.eval()
    @classmethod
    def from_artefacts(cls, art_dir=ART, device=None):
        device = device or ('cuda' if torch.cuda.is_available() else 'cpu')
        cfg = json.load(open(os.path.join(art_dir, 'config.json')))
        model = CNN(d=cfg['d'], hidden=cfg['hidden'], layers=cfg['layers'],
                    kernel=cfg['kernel'], dropout=cfg['dropout'])
        model.load_state_dict(torch.load(os.path.join(art_dir, 'cnn.pt'), map_location=device))
        model.to(device)
        return cls(model, cfg, device)
    def predict(self, sequence, pssm=None):
        if not sequence: return ''
        x = encode_sequence(sequence).unsqueeze(0).to(self.device)
        with torch.no_grad():
            idx = self.model(x).argmax(-1)[0].cpu().numpy()
        pred = ''.join(Q3_LABELS[int(i)] for i in idx)
        assert len(pred) == len(sequence)
        return pred
    def predict_from_fasta(self, fasta_path):
        return [(pid, seq, self.predict(seq)) for pid, seq in read_fasta(fasta_path)]
    @staticmethod
    def save_predictions(results, out_path, smooth=False, min_segment=3):
        n = 0
        with open(out_path, 'w', newline='') as f:
            w = csv.writer(f); w.writerow(['protein_id','position','residue','predicted_ss'])
            for pid, seq, pred in results:
                if smooth: pred = smooth_predictions(pred, min_segment)
                pred = collapse_q3(pred)
                for i, (aa, ss) in enumerate(zip(seq, pred), start=1):
                    w.writerow([pid, i, aa, ss]); n += 1
        return n

## 5. Run the frozen pipeline on the test set

In [ ]:
predictor = SecStructPredictor.from_artefacts(ART)
TEST_FASTA = os.path.join(ART, 'test.fasta') if os.path.exists(os.path.join(ART, 'test.fasta')) else 'test.fasta'
results = predictor.predict_from_fasta(TEST_FASTA)
n_raw = predictor.save_predictions(results, 'predictions.csv')
n_smo = predictor.save_predictions(results, 'predictions_smoothed.csv', smooth=True, min_segment=3)
print(f'{len(results)} chains -> predictions.csv ({n_raw} rows), predictions_smoothed.csv ({n_smo} rows)')

# provenance stamp (kept OUT of the graded CSV so the auto-check row count is exact)
stamp = {'model_file': f'{ART}/cnn.pt', 'input_fasta': TEST_FASTA,
         'trained_on_chains': CFG.get('num_chains'), 'val_q3': CFG.get('best_val_q3'),
         'seed': SEED, 'n_chains': len(results), 'n_residues': n_raw}
json.dump(stamp, open('predictions_stamp.json', 'w'), indent=2)
print('provenance:', stamp)

### CSV schema self-check (mirrors the auto-grader)

In [ ]:
import pandas as pd
def validate(csv_path, fasta):
    df = pd.read_csv(csv_path); fa = dict(read_fasta(fasta))
    problems = []
    if list(df.columns) != ['protein_id','position','residue','predicted_ss']:
        problems.append('bad columns')
    if len(df) != sum(len(s) for s in fa.values()):
        problems.append('row count != residues')
    if set(df.predicted_ss) - {'H','E','C'}: problems.append('non-Q3 labels')
    for pid, g in df.groupby('protein_id'):
        if list(g.position) != list(range(1, len(fa[pid])+1)): problems.append(f'{pid} positions')
        if ''.join(g.residue) != fa[pid]: problems.append(f'{pid} residues')
    return problems or 'OK'
print('predictions.csv         :', validate('predictions.csv', TEST_FASTA))
print('predictions_smoothed.csv:', validate('predictions_smoothed.csv', TEST_FASTA))

### Determinism — run twice, compare MD5

In [ ]:
def md5(p): return hashlib.md5(open(p,'rb').read()).hexdigest()
h1 = md5('predictions.csv')
predictor.save_predictions(predictor.predict_from_fasta(TEST_FASTA), 'predictions.csv')
h2 = md5('predictions.csv')
print('run1', h1); print('run2', h2); print('IDENTICAL:', h1 == h2)

## 6. Error analysis
Requires `artefacts/test_true.csv` (per-residue ground truth for the test chains).

In [ ]:
import matplotlib.pyplot as plt
from collections import defaultdict
Q3N = ['H','E','C']; IDX = {c:i for i,c in enumerate(Q3N)}
truth = pd.read_csv(os.path.join(ART, 'test_true.csv'))
pred_df = pd.read_csv('predictions.csv'); smo_df = pd.read_csv('predictions_smoothed.csv')
ch = {}
for pid, g in truth.groupby('protein_id', sort=False):
    g = g.sort_values('position'); ch[pid] = {'seq':''.join(g.residue),'true':''.join(g.true_ss)}
for name, d in (('raw',pred_df),('smooth',smo_df)):
    for pid, g in d.groupby('protein_id', sort=False):
        g = g.sort_values('position')
        if pid in ch: ch[pid][name] = ''.join(g.predicted_ss)
ch = {k:v for k,v in ch.items() if 'raw' in v}
T = ''.join(c['true'] for c in ch.values()); R = ''.join(c['raw'] for c in ch.values()); S=''.join(c['smooth'] for c in ch.values())
q3_raw = np.mean([a==b for a,b in zip(T,R)]); q3_smo = np.mean([a==b for a,b in zip(T,S)])
print(f'Q3 raw={q3_raw:.4f}  smoothed={q3_smo:.4f}  delta={q3_smo-q3_raw:+.4f}')

In [ ]:
from sklearn.metrics import confusion_matrix
cm = confusion_matrix([IDX[c] for c in T],[IDX[c] for c in R],labels=[0,1,2])
per_class = (cm.diagonal()/cm.sum(1).clip(min=1)).round(3)
print('confusion (rows=true H/E/C):'); print(cm); print('per-class acc:', dict(zip(Q3N, per_class)))
plt.figure(figsize=(3.4,3)); cmn = cm/cm.sum(1,keepdims=True).clip(min=1)
plt.imshow(cmn,cmap='Blues',vmin=0,vmax=1); plt.xticks(range(3),Q3N); plt.yticks(range(3),Q3N)
plt.xlabel('predicted'); plt.ylabel('true'); plt.title('Q3 confusion')
for i in range(3):
    for j in range(3): plt.text(j,i,cm[i,j],ha='center',va='center',color='white' if cmn[i,j]>.5 else 'black')
plt.tight_layout(); plt.show()

In [ ]:
def boundaries(ss):
    n=len(ss); return [ (i==0 or ss[i]!=ss[i-1]) or (i==n-1 or ss[i]!=ss[i+1]) for i in range(n) ]
bm = np.concatenate([boundaries(c['true']) for c in ch.values()])
ta, ra = np.array(list(T)), np.array(list(R))
print(f'boundary Q3={(ta[bm]==ra[bm]).mean():.4f}  interior Q3={(ta[~bm]==ra[~bm]).mean():.4f}')
# segment-length effect
buck=defaultdict(list)
for c in ch.values():
    t,p=c['true'],c['raw']; i=0
    while i<len(t):
        cls,j=t[i],i
        while j<len(t) and t[j]==cls: j+=1
        buck[min(j-i,15)].append(np.mean([p[k]==cls for k in range(i,j)])); i=j
xs=sorted(buck); plt.figure(figsize=(5,3)); plt.plot(xs,[np.mean(buck[k]) for k in xs],marker='o')
plt.xlabel('true segment length (15=15+)'); plt.ylabel('fraction correct'); plt.title('Accuracy vs segment length'); plt.grid(alpha=.3); plt.show()
# per-protein
pp={pid: np.mean([a==b for a,b in zip(c['true'],c['raw'])]) for pid,c in ch.items()}
hard=sorted(pp,key=pp.get)[:10]
print('10 hardest:', [(p, round(pp[p],3), len(ch[p]['seq'])) for p in hard])
plt.figure(figsize=(5,3)); plt.hist(list(pp.values()),bins=20,edgecolor='k'); plt.xlabel('per-protein Q3'); plt.ylabel('# chains'); plt.title(f'median {np.median(list(pp.values())):.3f}'); plt.show()

## 7. Biological inference (≥3 named test proteins)

In [ ]:
def comp(ss):
    n=max(len(ss),1); return {'H':round(ss.count('H')/n*100,1),'E':round(ss.count('E')/n*100,1),'C':round(ss.count('C')/n*100,1)}
def struct_class(c):
    H,E=c['H'],c['E']
    if H>50 and E<5: return 'all-alpha'
    if H<5 and E>30: return 'all-beta'
    if H<15 and E<10: return 'mostly coil / disordered'
    if 20<=H<=40 and 15<=E<=30: return 'alpha/beta'
    return 'mixed'
def long_runs(ss,label,m):
    runs=[];i=0
    while i<len(ss):
        if ss[i]==label:
            j=i
            while j<len(ss) and ss[j]==label: j+=1
            if j-i>=m: runs.append((i+1,j))
            i=j
        else: i+=1
    return runs
comps={pid: comp(c['raw']) for pid,c in ch.items()}
picks=[max(comps,key=lambda p:comps[p]['H']), max(comps,key=lambda p:comps[p]['E']), max(comps,key=lambda p:comps[p]['C'])]
for pid in dict.fromkeys(picks):
    pr=ch[pid]['raw']; tm=long_runs(pr,'H',20); idr=long_runs(pr,'C',30)
    print(f"{pid}: {comps[pid]} -> {struct_class(comps[pid])} | TM(>=20H)={len(tm)}{' [GPCR-like]' if len(tm)==7 else ''} | IDR(>30C)={len(idr)} | Q3={pp[pid]:.3f}")

## 8. Reproducibility checklist

In [ ]:
checks = [
  ('Model weights saved with the hyperparameter config that produced them', os.path.exists(f'{ART}/cnn.pt') and os.path.exists(f'{ART}/config.json')),
  ('Scaler/vectoriser fitted on training only, saved and loaded', True),  # sequence-only CNN: no external scaler; embedding is part of cnn.pt
  ('Random seed set (numpy, torch, python random) before split/inference', True),
  ('Train/val/test split saved & reproducible from seed', os.path.exists(f'{ART}/split.json')),
  ('Running the pipeline twice on the same input gives identical output', h1 == h2),
  ('predictions.csv stamped with which model file + dataset produced it', os.path.exists('predictions_stamp.json')),
]
for msg, ok in checks:
    print(f"[{'x' if ok else ' '}] {msg}")
assert all(ok for _, ok in checks), 'a reproducibility item failed'